# MambaCS — 1-Epoch Colab Debug Run

**Purpose:** Generate synthetic k-space HDF5 data and run a single training epoch to smoke-test the pipeline.

### Before running:
1. Upload your `MambaCS/` folder to Google Drive (e.g. `My Drive/MambaCS/`)
2. Set `REPO_PATH` in Cell 2 to wherever you put it
3. Runtime → Change runtime type → **GPU**
4. Run all cells top to bottom

In [ ]:
# ── Cell 1: Install missing dependencies ─────────────────────────────────────
!pip install -q einops wandb h5py

In [ ]:
# ── Cell 2: Mount Drive and set paths ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ⚠️  Change this to wherever your MambaCS folder lives in Drive
REPO_PATH  = '/content/drive/MyDrive/MambaCS'

# Where synthetic H5 data will be written (inside Drive so it persists)
DATA_ROOT  = '/content/drive/MyDrive/MambaCS_synth_data'
TRAIN_DIR  = os.path.join(DATA_ROOT, 'train')
VAL_DIR    = os.path.join(DATA_ROOT, 'val')

# Where checkpoints / metrics land
OUTPUT_DIR = '/content/drive/MyDrive/MambaCS_colab_experiments'

os.makedirs(TRAIN_DIR,  exist_ok=True)
os.makedirs(VAL_DIR,    exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Add repo to path so all imports resolve
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

print('Repo    :', REPO_PATH)
print('Train   :', TRAIN_DIR)
print('Val     :', VAL_DIR)
print('Outputs :', OUTPUT_DIR)
print('cwd     :', os.getcwd())

In [ ]:
# ── Cell 3: Generate synthetic complex k-space HDF5 files ────────────────────
#
# Format mirrors fastMRI singlecoil:
#   file.h5  →  dataset 'kspace'  shape (num_slices, H, W)  dtype complex64
#
# We use H=640, W=384.  dataset.py centre-crops W→320, keeping full height.

import numpy as np
import h5py

RNG = np.random.default_rng(42)

def make_phantom(H: int, W: int, rng) -> np.ndarray:
    """Simple multi-ellipse phantom — deterministic given rng state."""
    y, x = np.mgrid[-1:1:H*1j, -1:1:W*1j]   # normalised coords in [-1, 1]

    img  = np.zeros((H, W), dtype=np.float32)

    # outer body
    img += 0.8 * ((x / 0.85)**2 + (y / 0.95)**2 <= 1)

    # two lateral dark ellipses
    img -= 0.5 * (((x - 0.22) / 0.31)**2 + (y / 0.35)**2 <= 1)
    img -= 0.5 * (((x + 0.22) / 0.31)**2 + (y / 0.35)**2 <= 1)

    # small bright blobs (simulate lesions / structures)
    for _ in range(4):
        cx = rng.uniform(-0.5, 0.5)
        cy = rng.uniform(-0.5, 0.5)
        r  = rng.uniform(0.04, 0.12)
        amp = rng.uniform(0.1, 0.3)
        img += amp * ((x - cx)**2 + (y - cy)**2 <= r**2)

    return img.clip(0, 1)


def make_h5_file(path: str, num_slices: int, H: int = 640, W: int = 384):
    kspace = np.zeros((num_slices, H, W), dtype=np.complex64)
    for s in range(num_slices):
        phantom = make_phantom(H, W, RNG)
        # 2-D FFT → complex k-space  (same convention as fft_2d in dc.py)
        kspace[s] = np.fft.fft2(phantom).astype(np.complex64)
    with h5py.File(path, 'w') as f:
        f.create_dataset('kspace', data=kspace)
    print(f'  wrote {path}  shape={kspace.shape}  dtype={kspace.dtype}')


# ── Generate files ────────────────────────────────────────────────────────────
# 5 train files × 5 slices = 25 training samples   (enough to fill a few batches)
# 2 val   files × 5 slices = 10 validation samples

print('Generating training data...')
for i in range(5):
    make_h5_file(os.path.join(TRAIN_DIR, f'synth_train_{i:03d}.h5'), num_slices=5)

print('Generating validation data...')
for i in range(2):
    make_h5_file(os.path.join(VAL_DIR, f'synth_val_{i:03d}.h5'), num_slices=5)

print('Done.')

In [ ]:
# ── Cell 4: Quick sanity check — load one sample through the dataset ──────────
from dataset import H5MRIDataset
import torch

ds = H5MRIDataset(TRAIN_DIR, N=320, kspace_key='kspace')
sample = ds[0]
print('Dataset length :', len(ds))
print('Sample shape   :', sample.shape)   # expect [1, 640, 320]
print('Sample dtype   :', sample.dtype)   # expect torch.complex64
assert sample.shape == (1, 640, 320), f'Unexpected shape: {sample.shape}'
print('Dataset OK ✓')

In [ ]:
# ── Cell 5: Write a Colab-specific train_config.py ───────────────────────────
#
# We overwrite train_config.py inside the repo with a single experiment entry
# that points to the generated data, runs 1 epoch, and uses a tiny batch.

config_src = f'''"""Colab 1-epoch smoke-test config — auto-generated by colab_test_run.ipynb."""

EXPERIMENTS = [
    {{
        "prefix": "ColabTest",
        "name": "axial_rope_axial_1ep",

        # Data — generated synthetic H5 files
        "data_dir":     r"{TRAIN_DIR}",
        "val_data_dir": r"{VAL_DIR}",

        # Output
        "output_dir": r"{OUTPUT_DIR}",

        # Architecture (mirrors the Bunya experiment)
        "encoders":        ["axial", "axial", "axial"],
        "k_space_learning": True,
        "pos_emb_type":    "Rope-Axial",
        "attn_type":       "complex",

        # Training — minimal for a smoke-test
        "epochs":      1,
        "batch_size":  2,
        "num_workers": 2,
    }}
]
'''

config_path = os.path.join(REPO_PATH, 'train_config.py')
with open(config_path, 'w') as fh:
    fh.write(config_src)

print('Wrote', config_path)
print(config_src)

In [ ]:
# ── Cell 6: Run 1-epoch training ─────────────────────────────────────────────
#
# WANDB_MODE=offline  →  no login required, logs written locally

import subprocess, sys

result = subprocess.run(
    [sys.executable, 'train.py', '--exp_idx', '0'],
    cwd=REPO_PATH,
    env={**os.environ, 'WANDB_MODE': 'offline'},
    capture_output=False,   # print stdout/stderr live
)

print('\n─── return code:', result.returncode, '───')
if result.returncode != 0:
    print('Training failed — scroll up for the traceback.')
else:
    print('Training completed successfully.')

In [ ]:
# ── Cell 7: Inspect outputs ───────────────────────────────────────────────────
import json

exp_dir = os.path.join(OUTPUT_DIR, 'ColabTest_axial_rope_axial_1ep')
metrics_path = os.path.join(exp_dir, 'metrics.json')

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('Epoch metrics:')
    for m in metrics:
        print(' ', m)
else:
    print('metrics.json not found — training may have failed.')

print('\nFiles in experiment dir:')
if os.path.isdir(exp_dir):
    for fn in sorted(os.listdir(exp_dir)):
        print(' ', fn)
else:
    print('  (dir does not exist)')